# EDA учебной базы данных afisha

**Задача:** исследовать учебную базу `data-analyst-afisha` (5 таблиц) по трём направлениям:
1. Взаимосвязи между таблицами (PK, FK, типы связей)
2. Содержимое таблиц (соответствие описанию, объём данных)
3. Корректность данных (уникальность идентификаторов, пропуски, категориальные значения)

**Инструменты:** PostgreSQL (DBeaver) + pandas (проверка вторым инструментом).

Все SQL-запросы в ноутбуке выполнялись в DBeaver; для повторного прогона можно скопировать их в редактор DBeaver или настроить `sqlalchemy`-подключение (см. последнюю секцию).

## 1. Взаимосвязи между таблицами

### Первичные и внешние ключи

| Таблица | PK | FK |
|---|---|---|
| purchases | order_id | event_id → events |
| events | event_id | city_id → city, venue_id → venues |
| city | city_id | region_id → regions |
| venues | venue_id | — |
| regions | region_id | — |

### Типы связей

Все связи — **«один ко многим» (1:M)**. Связей «один к одному» и «многие ко многим» нет:
FK размещены на стороне «многих», PK каждой таблицы уникален.

```
regions —< city —< events —< purchases
venues  —< events
```

### Подтверждение связей данными

In [ ]:
-- Уникальность PK purchases: order_id
SELECT COUNT(*) AS total_rows,
       COUNT(order_id) AS non_null_ids,
       COUNT(DISTINCT order_id) AS unique_ids
FROM afisha.purchases;
-- Результат: 292034 = 292034 = 292034 → order_id уникален, один заказ = одна строка

In [ ]:
-- FK event_id повторяется: связь events 1:M purchases
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT event_id) AS unique_events
FROM afisha.purchases;
-- Результат: 292034 строк / 22484 мероприятия ≈ 13 покупок на мероприятие

## 2. Содержимое таблиц

| Таблица | Строк |
|---|---|
| purchases | 292 034 заказа |
| events | 22 484 мероприятия |
| city | 353 города |
| regions | 81 регион |
| уникальные пользователи | 22 000 |

### Временной охват

`created_dt_msk`: **2024-06-01 → 2024-10-31** (5 месяцев).
`MAX - MIN` = 152 дня, `COUNT(DISTINCT created_dt_msk)` = 153 → данные есть за каждый день, покрытие полное, без пропусков.

In [ ]:
SELECT MIN(created_dt_msk),
       MAX(created_dt_msk),
       COUNT(DISTINCT created_dt_msk),
       MAX(created_dt_msk) - MIN(created_dt_msk)
FROM afisha.purchases;
-- Результат: 2024-06-01 | 2024-10-31 | 153 | 152 days

## 3. Корректность данных

### 3.1 Уникальность идентификаторов

- `order_id` (purchases): дубликатов нет (292 034 = 292 034)
- `event_id` (events): **873 строки полностью идентичны за исключением id** — проверено через `ROW_NUMBER()`: отличия только в `event_id`. Интерпретация: одинаковые мероприятия, проходившие в разные даты. Помнить при аналитике: одно мероприятие может иметь несколько `event_id`.

### 3.2 Пропуски (NULL)

Проверка автоматизирована: запрос сгенерирован через `information_schema.columns` (string_agg + UNION ALL), чтобы не перечислять колонки вручную.

In [ ]:
-- Проверка дубликатов в events: строки, идентичные по всем колонкам, кроме event_id
SELECT COUNT(*) AS duplicate_rows
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY event_name_code, event_type_description,
                              event_type_main, organizers, city_id, venue_id
                              ORDER BY event_id) AS rn
    FROM afisha.events
) t
WHERE rn > 1;
-- Результат: 873 строки — по всем колонкам, кроме event_id, идентичны.
-- Интерпретация: одинаковые мероприятия, проходившие в разные даты (либо дубль в справочнике).

In [ ]:
-- Шаг 1: генератор запроса NULL-проверок по ВСЕМ таблицам
SELECT string_agg(
    format(
        'SELECT %I AS table_name, %L AS column_name, COUNT(*) - COUNT(%I) AS null_count FROM %I.%I',
        table_name, column_name, column_name, table_schema, table_name
    ),
    ' UNION ALL '
) AS generated_sql
FROM information_schema.columns
WHERE table_schema = 'afisha'
  AND table_name IN ('purchases', 'events', 'venues', 'city', 'regions');

-- Шаг 2: скопировать текст из ячейки generated_sql и выполнить его.
-- Результат: все 30 проверок вернули null_count = 0 → пропусков нет ни в одной колонке.

**Результат:** NULL-значений нет ни в одной колонке всех пяти таблиц. `COUNT(*)` и `COUNT(колонка)` везде эквивалентны.

### 3.3 Категориальные значения

In [ ]:
-- cinema_circuit: 'нет' 290632 | Другое 1263 | КиноСити 124 | Киномакс 7 | Москино 7 | ЦентрФильм 1
SELECT cinema_circuit, COUNT(*)
FROM afisha.purchases
GROUP BY cinema_circuit
ORDER BY COUNT(*) DESC;

In [ ]:
-- currency_code: 2 валюты
SELECT currency_code, COUNT(*)
FROM afisha.purchases
GROUP BY currency_code;
-- rub 286961 | kzt 5073
-- ⚠️ Разные масштабы цен: суммы total/revenue считать только в разрезе валют

In [ ]:
-- device_type_canonical: mobile 232679 | desktop 58170 | tablet 1180 | other 2 | tv 3
SELECT device_type_canonical, COUNT(*)
FROM afisha.purchases
GROUP BY device_type_canonical
ORDER BY COUNT(*) DESC;

In [ ]:
-- age_limit: 0 (61731) | 6 (52403) | 12 (62861) | 16 (78864) | 18 (36175) — только валидные значения
SELECT age_limit, COUNT(*)
FROM afisha.purchases
GROUP BY age_limit
ORDER BY age_limit;

Категориальные значения написаны корректно: опечаток и нестандартных значений не выявлено. `service_name` — 36 билетных операторов.

## 4. Аномалии и их разбор

### 4.1 Заказы с нулевой выручкой → бесплатные мероприятия

**Цепочка доказательства:**

1. 5 772 заказа (≈2%) имеют `revenue = 0`, при этом **у всех `tickets_count > 0`** → исключаем «пустые» заказы
2. 295 мероприятий имеют нулевую суммарную выручку, и **ни у одного нет платного заказа** → нулевая выручка — свойство мероприятия целиком
3. Отрицательная выручка в базе копеечная (мин. −90) → погасить платный заказ до нуля невозможно → «скрытые возвраты» исключены математически

**Вывод: 295 полностью бесплатных мероприятий, ≈20 заказов на каждое (5 772 / 295). Гипотеза возвратов/отмен не подтвердилась.**

In [ ]:
-- Шаг 1: у zero-revenue заказов есть билеты?
SELECT tickets_count, COUNT(*)
FROM afisha.purchases
WHERE revenue = 0
GROUP BY tickets_count
ORDER BY tickets_count;
-- Все 5772 заказа имеют tickets_count >= 1 (в основном 1–6 билетов)

In [ ]:
-- Шаг 2: мероприятия с нулевой суммарной выручкой — есть ли среди них «смешанные»?
SELECT
    COUNT(*) AS zero_events,
    SUM(flag_0) AS events_with_mixed_revenue
FROM (
    SELECT
        p.event_id,
        MAX(CASE WHEN p.revenue > 0 THEN 1 ELSE 0 END) AS flag_0
    FROM afisha.purchases p
    GROUP BY p.event_id
    HAVING SUM(p.revenue) = 0
) t;
-- Результат: 295 | 0 → все zero-мероприятия полностью бесплатные

Приём из этого блока: `MAX(CASE WHEN условие THEN 1 ELSE 0 END)` = «условие выполнилось хотя бы раз в группе». Без него нельзя отличить «настоящие нули» от нулей, полученных взаимным погашением плюс-минус при возвратах.

### 4.2 Отрицательная выручка

381 заказ с `revenue < 0`, минимум −90, в основном несколько рублей. Гипотеза: комиссия/корректировка биллинга. Малые значения не влияют на вывод о бесплатных мероприятиях.

In [ ]:
SELECT
    p.revenue, p.total, p.tickets_count, p.currency_code, p.service_name
FROM afisha.purchases p
WHERE p.revenue < 0
ORDER BY p.revenue
LIMIT 20;
-- TODO: изучить «паспорт» минусовых заказов (связь revenue и total, зависимость от оператора)

### 4.3 Неуникальные имена городов

Город **«Глинополье» встречается в `city` дважды**, в разных регионах → группировать по городу только по `city_id`, не по `city_name`.

## 5. Открытые проверки (TODO)

### 5.1 Referential integrity (целостность FK)

In [ ]:
-- Шаблон: LEFT JOIN + IS NULL, ожидается 0
SELECT COUNT(*) AS broken_refs
FROM afisha.purchases p
LEFT JOIN afisha.events e ON p.event_id = e.event_id
WHERE e.event_id IS NULL;
-- TODO: аналогично для events.city_id → city, events.venue_id → venues, city.region_id → regions

### 5.2 Какие типы мероприятий бесплатные

Финальный штрих к пункту 4.1: разбить 295 бесплатных мероприятий по `event_type_main`.

In [ ]:
SELECT
    e.event_type_main,
    COUNT(*) AS free_events,
    SUM(cnt) AS free_orders
FROM (
    SELECT p.event_id, COUNT(*) AS cnt
    FROM afisha.purchases p
    GROUP BY p.event_id
    HAVING SUM(p.revenue) = 0
) t
JOIN afisha.events e ON t.event_id = e.event_id
GROUP BY e.event_type_main
ORDER BY free_orders DESC;
-- TODO: интерпретировать, какие типы мероприятий бесплатно

### 5.3 Суммы в разрезе валют

При любой агрегации `total`/`revenue` в одну цифру — учитывать смешение валют (rub 286 961 заказ / kzt 5 073) либо нормализовать курсом.

### 5.4 Проверка purchases в pandas

Подтверждение SQL-выводов вторым инструментом. Вариант подключения к PostgreSQL из pandas:

In [ ]:
# pip install sqlalchemy psycopg2-binary pandas
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('postgresql+psycopg2://<user>:<password>@<host>:5432/<database>')

df = pd.read_sql('SELECT * FROM afisha.purchases', engine)

In [ ]:
# Проверки качества данных
print('Полные дубликаты:', df.duplicated().sum())        # ожидается 0
print('Пропуски по колонкам:')
print(df.isna().sum())                                   # ожидается: все нули
print('order_id уникален:', df['order_id'].is_unique)     # ожидается True
print('Дубли event_id (бизнес-логика 1:M):', df['event_id'].duplicated().sum())
df['total'].describe()                                   # выбросы и распределение

## 6. Выводы

1. **Схема целостная:** пять таблиц, все связи 1:M, ключи соответствуют описанию. Связи подтверждены данными (уникальность `order_id`, повторяемость `event_id`).
2. **Пропусков нет ни в одной колонке** — проверено автоматически через `information_schema`.
3. **Категориальные данные корректны:** без опечаток; 2 валюты, 5 возрастных ограничений, 36 операторов.
4. **Временной охват:** 5 месяцев (июнь–октябрь 2024), полное покрытие по дням.
5. **295 мероприятий полностью бесплатные** (≈2% заказов) — доказано трёхшаговой проверкой; возвраты/отмены исключены.
6. **Учёт в дальнейшей аналитике:** смешение валют (rub/kzt), города-омонимы в разных регионах (группировать по `city_id`), дубли мероприятий в `events` (873 строки, отличия только в id), копеечная отрицательная выручка (381 заказ, макс. −90).